# 🚀 Biến Google Colab Thành Server Học Tập CI/CD
Notebook này thiết lập tự động môi trường trên Google Colab để:
1. **Gói 1 (CI)**: Làm **GitHub Actions Self-Hosted Runner** (Chạy test, build, tận dụng GPU/CPU miễn phí).
2. **Gói 2 (CD)**: Làm **Staging/Deployment Server** với Docker, OpenSSH và Tunnel công khai (Cloudflare/Tailscale).

### Bước 1: Giữ phiên làm việc (Keep-Alive JavaScript)
Chạy cell dưới đây để bật script JavaScript giữ kết nối trình duyệt tránh bị Colab ngắt kết nối khi chuyển tab.

In [ ]:
from IPython.display import display, Javascript

def keep_alive_js():
    js_code = '''
    function ClickConnect(){
        console.log("[Keep-Alive] Pinging Colab connection...");
        var btn = document.querySelector("colab-connect-button");
        if (btn) btn.click();
    }
    setInterval(ClickConnect, 60000);
    console.log("[Keep-Alive] Script đã kích hoạt (chu kỳ 60s)");
    '''
    display(Javascript(js_code))

keep_alive_js()

### Bước 2: (Tùy chọn) Gắn kết Google Drive để lưu trữ dữ liệu vĩnh viễn

In [ ]:
# Bỏ comment dòng dưới nếu muốn mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/colab_server

### Bước 3: Tải mã nguồn bộ công cụ `Colab2Server`

In [ ]:
import os

WORKSPACE_DIR = "/content/Colab2Server"
REPO_URL = "https://github.com/giangpvg/Colab2Server.git"

if not os.path.exists(WORKSPACE_DIR):
    print(">>> Đang clone mã nguồn Colab2Server từ GitHub...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print(">>> Thư mục đã tồn tại, đang kéo mã nguồn mới nhất...")
    !git -C {WORKSPACE_DIR} pull origin main 2>/dev/null || true

%cd {WORKSPACE_DIR}
!chmod +x scripts/*.sh scripts/*.py 2>/dev/null || true
print("
>>> Danh sách script đã sẵn sàng:")
!ls -la scripts/


### Bước 4: Cấu hình thông số (Điền thông tin vào form bên phải ➡️)

In [ ]:
#@title ⚙️ Bảng Cấu Hình Dịch Vụ { form-width: "60%" }

# --- 1. Gói CI: GitHub Actions Runner ---
ENABLE_RUNNER = True #@param {type:"boolean"}
GITHUB_REPO_URL = "https://github.com/USERNAME/REPO_NAME" #@param {type:"string"}
RUNNER_TOKEN = "YOUR_GITHUB_RUNNER_REGISTRATION_TOKEN" #@param {type:"string"}
RUNNER_NAME = "colab-runner-01" #@param {type:"string"}
RUNNER_LABELS = "colab,gpu,cuda,staging" #@param {type:"string"}

# --- 2. Gói CD: Docker & OpenSSH Server ---
ENABLE_DOCKER = True #@param {type:"boolean"}
ENABLE_SSH = True #@param {type:"boolean"}
SSH_ROOT_PASSWORD = "colab123456" #@param {type:"string"}
SSH_PUBLIC_KEY = "" #@param {type:"string"}

# --- 3. Mạng & Tunnel ---
TUNNEL_MODE = "cloudflare-quick" #@param ["none", "cloudflare-quick", "cloudflare-token", "tailscale"]
TUNNEL_SECRET = "" #@param {type:"string"}

print("Đã lưu thông số cấu hình!")

### Bước 5: Kích hoạt toàn bộ dịch vụ (1-Click Run)

In [ ]:
# 1. Khởi tạo hệ thống cơ bản
!bash scripts/setup_system.sh

# 2. Kích hoạt Docker
if ENABLE_DOCKER:
    !bash scripts/setup_docker.sh

# 3. Kích hoạt SSH Server
if ENABLE_SSH:
    !bash scripts/setup_ssh.sh "{SSH_ROOT_PASSWORD}" "{SSH_PUBLIC_KEY}"

# 4. Kích hoạt Tunnel mạng
if TUNNEL_MODE != "none":
    !bash scripts/setup_tunnel.sh "{TUNNEL_MODE}" "{TUNNEL_SECRET}"

# 5. Đăng ký & Kích hoạt GitHub Actions Runner
if ENABLE_RUNNER and "USERNAME" not in GITHUB_REPO_URL and RUNNER_TOKEN != "YOUR_GITHUB_RUNNER_REGISTRATION_TOKEN":
    !bash scripts/setup_runner.sh "{GITHUB_REPO_URL}" "{RUNNER_TOKEN}" "{RUNNER_NAME}" "{RUNNER_LABELS}"
else:
    print("ℹ️ Bỏ qua kích hoạt Runner (chưa cấu hình GITHUB_REPO_URL hoặc RUNNER_TOKEN hợp lệ).")

### Bước 6: Giám sát tài nguyên và Duy trì phiên (Keep-Alive Dashboard)
Chạy cell này để hiển thị dashboard theo dõi RAM, GPU, CPU và trạng thái các dịch vụ định kỳ.

In [ ]:
!python3 scripts/keep_alive.py